# MLP 4x100 GELU + Dropout + L2

Arquitectura fija basada en el mejor resultado del notebook de busqueda de hiperparametros.

Config:

- 4 capas ocultas
- 100 neuronas por capa
- activacion GELU
- Dropout `0.20`
- L2 `1e-5`
- Adam con learning rate `1e-4`
- batch size `256`
- EarlyStopping con `restore_best_weights=True`

Entrena esta arquitectura para las 16 combinaciones de ventanas y guarda resultados en `data/mlp/`.

Este notebook registra cada celda del grid en MLflow igual que los notebooks `01` a `06`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "util.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "util.py").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import regularizers
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.layers import Dense, Dropout, Input
from keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error

from util import (
    get_train_test,
    RANDOM_SEED,
    compare_to_benchmark,
    plot_benchmark_comparison,
    configure_mlflow,
    log_keras_grid_run,
)

keras.utils.set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
MODEL_NAME = "mlp_4x100_gelu_dropout_l2"
EXPERIMENT_NAME = "MLP"
LOG_MODEL_ARTIFACT = False
ARCHITECTURE_PARAMS = {
    "hidden_layers": 4,
    "neurons": 100,
    "activation": "gelu",
    "regularization": "dropout_l2",
    "dropout_rate": 0.20,
    "l2": 1e-5,
    "batch_norm": False,
    "early_stopping": True,
    "patience": 20,
    "min_delta": 1e-6,
}

mlflow = configure_mlflow(EXPERIMENT_NAME)
RESULTS_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}.csv"
HISTORY_PATH = PROJECT_ROOT / "data" / "mlp" / f"{MODEL_NAME}_history.csv"

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

EPOCHS = 500
BATCH_SIZE = 256
LEARNING_RATE = 1e-4
VALIDATION_SPLIT = 0.1
PATIENCE = 20
MIN_DELTA = 1e-6

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS_PATH)
print("history:", HISTORY_PATH)

## Definicion de la red

In [ ]:
def build_model(input_dim, output_dim):
    l2_reg = regularizers.l2(1e-5)

    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    model.add(Dense(100, activation="gelu", kernel_regularizer=l2_reg))
    model.add(Dropout(0.20))
    model.add(Dense(100, activation="gelu", kernel_regularizer=l2_reg))
    model.add(Dropout(0.20))
    model.add(Dense(100, activation="gelu", kernel_regularizer=l2_reg))
    model.add(Dropout(0.20))
    model.add(Dense(100, activation="gelu", kernel_regularizer=l2_reg))
    model.add(Dropout(0.20))
    model.add(Dense(output_dim))
    model.compile(loss="mean_absolute_error", optimizer=Adam(learning_rate=LEARNING_RATE))
    return model


def make_early_stopping():
    return EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        restore_best_weights=True,
    )


## Entrenamiento en grid de ventanas

In [ ]:
results = []
history_rows = []

for in_w in input_windows:
    for out_w in output_windows:
        d = get_train_test(input_window_size=in_w, output_window_size=out_w)

        X_train = d.X_train.reshape(d.X_train.shape[0], -1)
        X_test = d.X_test.reshape(d.X_test.shape[0], -1)

        keras.utils.set_random_seed(RANDOM_SEED)
        model = build_model(input_dim=X_train.shape[1], output_dim=d.y_train.shape[1])

        hist = model.fit(
            X_train,
            d.y_train,
            validation_split=VALIDATION_SPLIT,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            # callbacks=[make_early_stopping()],
            verbose=0,
            shuffle=False,
        )

        y_pred_train = model.predict(X_train, verbose=0)
        y_pred_test = model.predict(X_test, verbose=0)

        val_losses = np.asarray(hist.history["val_loss"], dtype=float)
        row = {
            "model_name": MODEL_NAME,
            "input_window": in_w,
            "output_window": out_w,
            "MAE_train": mean_absolute_error(d.y_train, y_pred_train),
            "MAE_val": float(val_losses.min()),
            "MAE_test": mean_absolute_error(d.y_test, y_pred_test),
            "epochs": len(hist.history["loss"]),
            "n_params": model.count_params(),
        }
        results.append(row)

        for epoch, (loss, val_loss) in enumerate(
            zip(hist.history["loss"], hist.history["val_loss"]), start=1
        ):
            history_rows.append({
                "model_name": MODEL_NAME,
                "input_window": in_w,
                "output_window": out_w,
                "epoch": epoch,
                "loss": loss,
                "val_loss": val_loss,
            })

        run_name = f"{MODEL_NAME}_input{in_w}_output{out_w}"
        log_keras_grid_run(
            mlflow=mlflow,
            model=model,
            history=hist,
            run_name=run_name,
            model_name=MODEL_NAME,
            input_window=in_w,
            output_window=out_w,
            metrics_row=row,
            train_shape=X_train.shape,
            test_shape=X_test.shape,
            output_dim=d.y_train.shape[1],
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            validation_split=VALIDATION_SPLIT,
            extra_params=ARCHITECTURE_PARAMS,
            log_model_artifact=LOG_MODEL_ARTIFACT,
        )

        results_df = pd.DataFrame(results)
        history_df = pd.DataFrame(history_rows)
        results_df.to_csv(RESULTS_PATH, index=False)
        history_df.to_csv(HISTORY_PATH, index=False)

        print(
            f"input={in_w:>3} output={out_w:>3} | "
            f"train={row['MAE_train']:.6f} val={row['MAE_val']:.6f} "
            f"test={row['MAE_test']:.6f} epochs={row['epochs']} "
            f"params={row['n_params']}"
        )

results_df = pd.DataFrame(results)
history_df = pd.DataFrame(history_rows)
results_df

## Matrices de error

In [ ]:
mae_train_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_train")
mae_val_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_val")
mae_test_matrix = results_df.pivot(index="input_window", columns="output_window", values="MAE_test")

print("MAE train")
display(mae_train_matrix)
print("MAE val")
display(mae_val_matrix)
print("MAE test")
display(mae_test_matrix)

## Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices = [
    (mae_train_matrix, "MAE train"),
    (mae_val_matrix, "MAE val"),
    (mae_test_matrix, "MAE test"),
]
vmin = min(matrix.values.min() for matrix, _ in matrices)
vmax = max(matrix.values.max() for matrix, _ in matrices)

for ax, (matrix, title) in zip(axes, matrices):
    im = ax.imshow(matrix.values, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(matrix.columns)))
    ax.set_xticklabels(matrix.columns)
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    ax.set_xlabel("Ventana salida")
    ax.set_ylabel("Ventana entrada")
    ax.set_title(f"{title} - {MODEL_NAME}")
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix.values[i, j]:.4f}", ha="center", va="center", color="white", fontsize=9)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## Curvas de aprendizaje

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14), sharey=False)
axes = axes.ravel()

for ax, ((in_w, out_w), group) in zip(
    axes,
    history_df.groupby(["input_window", "output_window"], sort=True),
):
    ax.plot(group["epoch"], group["loss"], label="train")
    ax.plot(group["epoch"], group["val_loss"], label="val")
    ax.set_title(f"in={in_w}, out={out_w}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MAE")
    ax.grid(True, alpha=0.3)

axes[0].legend()
plt.suptitle(f"Curvas de aprendizaje - {MODEL_NAME}", y=1.02)
plt.tight_layout()
plt.show()

## Comparacion contra regresion lineal

In [ ]:
comparison = compare_to_benchmark(results_df, benchmark="lr_benchmark")
display(comparison)

plot_benchmark_comparison(results_df, benchmark="lr_benchmark", model_name=MODEL_NAME)
plt.show()